[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BMGLab/BFB/blob/main/W07_RNA_counts__normalization_and_defensible_expression_claims.ipynb)

# Week 07 | RNA counts, normalization and defensible expression claims

**Core practical: 45 minutes.** Keep sample identity intact and distinguish count normalization from inference.

No paid AI tool, local installation or external dataset download is required. Open it in Colab with the badge above, then **File > Save a copy in Drive** before you start so your work is kept. Run the cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

## Before running (5 min)
A gene may produce several transcripts; counts depend on what features and assignment rules were used.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Who is submitting (1 min)
Fill in your **full name** and your **Ege student number**, then run the cell. It refuses to
continue if either is missing or malformed, so a mistyped digit is caught here — in the room,
where it takes ten seconds to fix — rather than after the deadline.

Your `COURSE_ID` above still deals your dataset. This is only about attributing the work to you.

In [ ]:
import os
if not os.path.exists("bib_colab.py"):
    !curl -sfO https://raw.githubusercontent.com/BMGLab/BFB/main/bib_colab.py
import bib_colab as bib

A = bib.start("W07")
A.whoami(
    name="",          # your full name, e.g. "Ayşe Gül Öztürk"
    student_no="",    # your Ege student number, digits only
    section="EN",     # "EN" or "TR"
)

### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Inspect the real two-gene excerpt and deliberately shuffled sample metadata.
2. Remove the documented fb suffix and explicitly match sample names.
3. Calculate a separate synthetic TPM example.
4. State which analyses the two-gene excerpt cannot support.

In [ ]:
# REAL EXCERPT: first two rows printed in the DESeq2 release vignette, accessed 2026-09-14.
# Only two genes: NOT a complete library and NOT adequate for a differential-expression analysis.
samples = ["untreated1","untreated2","untreated3","untreated4","treated1","treated2","treated3"]
counts = {"FBgn0000003": [0,0,0,0,0,0,1], "FBgn0000008": [92,161,76,70,140,88,70]}
metadata_raw = [("treated1fb","treated","single-read"),("treated2fb","treated","paired-end"),
 ("treated3fb","treated","paired-end"),("untreated1fb","untreated","single-read"),
 ("untreated2fb","untreated","single-read"),("untreated3fb","untreated","paired-end"),
 ("untreated4fb","untreated","paired-end")]
metadata = {name.removesuffix("fb"): {"condition":cond,"type":typ} for name,cond,typ in metadata_raw}
assert set(samples) == set(metadata)
aligned = [metadata[s] for s in samples]
print("Matched columns:", list(zip(samples, aligned)))
# SYNTHETIC separate two-gene library, complete only within this toy example.
toy_counts, lengths_kb = [100,200], [1.0,2.0]
rpk = [c/L for c,L in zip(toy_counts,lengths_kb)]
tpm = [x/sum(rpk)*1_000_000 for x in rpk]
assert abs(sum(tpm) - 1_000_000) < 1e-8
print("Synthetic TPM:", tpm)
RESULTS = {"data_status": "REAL 2-gene excerpt for metadata exercise; SEPARATE synthetic TPM example",
           "matched_conditions": [x["condition"] for x in aligned], "toy_TPM": tpm}

## Explain the evidence (10 min)
**Q1.** Explain why attaching condition labels by row order would be wrong here.

**Q2.** Why do counts 100 and 200 yield equal TPM in the synthetic example?

**Q3.** May the real two-gene excerpt support a valid DESeq2 analysis or be treated as the full library? Explain.

In [ ]:
PREDICTION = ""  # Fill before the analysis.
RESPONSES = {"Q1": "", "Q2": "", "Q3": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, then **Runtime > Restart session and run all** so every number in
the notebook is the one your answers describe.

Then run the cell below. It checks that nothing is missing, prints a receipt code, and sends this
notebook straight to your instructor. There is nothing to download and nothing to upload.

A completion check looks for the presence of your responses, not for scientific correctness. If
the upload fails, the cell prints your receipt code and what to do instead — follow it before you
leave. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":7, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W07_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

# --- submit -------------------------------------------------------------
A.answers(RESPONSES,
          prediction=PREDICTION,
          check=CHECK_PERFORMED,
          disclosure=AI_DISCLOSURE,
          results=RESULTS)
A.check()
A.submit()

## Paper / device-free route
Match seven printed sample labels and conditions by name. Divide100/1 and200/2, then normalize each100/200x1million.

## Optional extension
Optional: inspect median-of-ratios normalization and effective lengths; no dispersion derivation is assessed.

## Sources
- [S09] Love, Huber and Anders (2014). Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. https://doi.org/10.1186/s13059-014-0550-8
- [S10] DESeq2: Analyzing RNA-seq data with DESeq2, release vignette. https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html
- [S25] Huber and Reyes. pasilla data package; Brooks et al. (2011) experiment. https://bioconductor.org/packages/release/data/experiment/html/pasilla.html